In [1]:
#!/usr/bin/env python3
"""
Secure Svalbard Satellite Image Fetcher
Uses environment variables for API credentials
"""

import os
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from sentinelhub import (
    SHConfig,
    DataCollection,
    SentinelHubRequest,
    BBox,
    CRS,
    MimeType,
)

def setup_config():
    """Setup Sentinel Hub configuration from environment variables"""
    client_id = os.getenv('SENTINELHUB_CLIENT_ID')
    client_secret = os.getenv('SENTINELHUB_CLIENT_SECRET')
    
    if not client_id or not client_secret:
        print("Error: Missing environment variables")
        print("Set these environment variables:")
        print("  export SENTINELHUB_CLIENT_ID='your_client_id'")
        print("  export SENTINELHUB_CLIENT_SECRET='your_client_secret'")
        return None
    
    config = SHConfig()
    config.sh_client_id = client_id
    config.sh_client_secret = client_secret
    config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    config.sh_base_url = "https://sh.dataspace.copernicus.eu"
    
    return config

def get_svalbard_image():
    """Download and display Svalbard satellite image"""
    
    # Setup
    config = setup_config()
    if not config:
        return
    
    # Svalbard bounding box
    svalbard_bbox = BBox(bbox=[10.0, 76.0, 35.0, 81.0], crs=CRS.WGS84)
    size = (2000, 1500)  # Width x Height, under 2500 pixel limit
    
    # Time range (last 30 days)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)
    time_interval = (start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
    
    # True color RGB script
    evalscript = """
    //VERSION=3
    function setup() {
        return {
            input: [{bands: ["B02", "B03", "B04"]}],
            output: {bands: 3}
        };
    }
    function evaluatePixel(sample) {
        return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02];
    }
    """
    
    print(f"Downloading Svalbard image ({size[0]}x{size[1]} pixels)...")
    
    # Create request
    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A.define_from(
                    name="s2l2a", service_url="https://sh.dataspace.copernicus.eu"
                ),
                time_interval=time_interval,
                other_args={"dataFilter": {"mosaickingOrder": "leastCC"}},
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
        bbox=svalbard_bbox,
        size=size,
        config=config,
    )
    
    # Download
    images = request.get_data()
    
    if images and len(images) > 0:
        image = images[0]
        print(f"Downloaded image: {image.shape}")
        
        # Display
        plt.figure(figsize=(12, 8))
        plt.imshow(image)
        plt.title(f"Svalbard Satellite Image\nSentinel-2 ({time_interval[0]} to {time_interval[1]})")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Save
        filename = f"svalbard_{datetime.now().strftime('%Y%m%d')}.png"
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        print(f"Saved: {filename}")
        
    else:
        print("No image data received")

if __name__ == "__main__":
    get_svalbard_image()

Error: Missing environment variables
Set these environment variables:
  export SENTINELHUB_CLIENT_ID='your_client_id'
  export SENTINELHUB_CLIENT_SECRET='your_client_secret'


In [2]:
#!/usr/bin/env python3
"""
Secure Svalbard Satellite Image Fetcher
Fetches both Sentinel-2 (optical) and Sentinel-1 (SAR) imagery
Uses environment variables for API credentials
"""

import os
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from sentinelhub import (
    SHConfig,
    DataCollection,
    SentinelHubRequest,
    BBox,
    CRS,
    MimeType,
)


def setup_config():
    """Setup Sentinel Hub configuration from environment variables"""
    client_id = os.getenv("SENTINELHUB_CLIENT_ID")
    client_secret = os.getenv("SENTINELHUB_CLIENT_SECRET")

    if not client_id or not client_secret:
        print("Error: Missing environment variables")
        print("Set these environment variables:")
        print("  export SENTINELHUB_CLIENT_ID='your_client_id'")
        print("  export SENTINELHUB_CLIENT_SECRET='your_client_secret'")
        return None

    config = SHConfig()
    config.sh_client_id = client_id
    config.sh_client_secret = client_secret
    config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    config.sh_base_url = "https://sh.dataspace.copernicus.eu"

    return config


def get_image(data_collection, evalscript, label, time_interval, config, bbox, size):
    """Generic function to download and display image"""
    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=data_collection,
                time_interval=time_interval,
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
        bbox=bbox,
        size=size,
        config=config,
    )

    images = request.get_data()

    if images and len(images) > 0:
        image = images[0]
        print(f"{label}: Downloaded image {image.shape}")

        # Display
        plt.figure(figsize=(12, 8))
        if image.shape[-1] == 1:  # SAR grayscale
            plt.imshow(image[:, :, 0], cmap="gray")
        else:  # Optical RGB
            plt.imshow(image)
        plt.title(f"{label} ({time_interval[0]} to {time_interval[1]})")
        plt.axis("off")
        plt.tight_layout()
        filename = f"{label.lower().replace(' ', '_')}_{datetime.now().strftime('%Y%m%d')}.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        print(f"{label}: Saved {filename}")
        plt.show()
    else:
        print(f"{label}: No image data received")


def main():
    config = setup_config()
    if not config:
        return

    # Area of interest: Svalbard
    svalbard_bbox = BBox(bbox=[10.0, 76.0, 35.0, 81.0], crs=CRS.WGS84)
    size = (2000, 1500)  # under 2500 px limit

    # Time range (last 30 days)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=30)
    time_interval = (start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d"))

    # Sentinel-2 RGB evalscript
    evalscript_s2 = """
    //VERSION=3
    function setup() {
        return {
            input: [{bands: ["B02", "B03", "B04"]}],
            output: {bands: 3}
        };
    }
    function evaluatePixel(sample) {
        return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02];
    }
    """

    # Sentinel-1 VV polarization evalscript
    evalscript_s1 = """
    //VERSION=3
    function setup() {
      return {
        input: [{bands: ["VV"], units: "dB"}],
        output: {bands: 1}
      };
    }
    function evaluatePixel(sample) {
      return [sample.VV];
    }
    """

    print("Downloading Sentinel-2 L2A (optical)...")
    get_image(
        DataCollection.SENTINEL2_L2A,
        evalscript_s2,
        "Sentinel-2 L2A",
        time_interval,
        config,
        svalbard_bbox,
        size,
    )

    print("Downloading Sentinel-1 IW (SAR)...")
    get_image(
        DataCollection.SENTINEL1_IW,
        evalscript_s1,
        "Sentinel-1 IW",
        time_interval,
        config,
        svalbard_bbox,
        size,
    )


if __name__ == "__main__":
    main()

Error: Missing environment variables
Set these environment variables:
  export SENTINELHUB_CLIENT_ID='your_client_id'
  export SENTINELHUB_CLIENT_SECRET='your_client_secret'


In [ ]:
#!/usr/bin/env python3
"""
BarentsWatch Arctic Vessel Tracker
Fetches all vessels and filters for Arctic region (above 65°N)
"""

import requests
import json
from datetime import datetime

# Your proven working credentials
CLIENT_ID = "henrikformoe@gmail.com:ArcticShadowTrackerAIS"
CLIENT_SECRET = "Xw5yCEXT5gMi5PJEKEW6"
SCOPE = "ais"

def get_access_token():
    """Get access token using proven credentials"""
    print("Getting access token...")
    
    token_url = "https://id.barentswatch.no/connect/token"
    
    data = {
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope': SCOPE,
        'grant_type': 'client_credentials'
    }
    
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    
    try:
        response = requests.post(token_url, data=data, headers=headers)
        response.raise_for_status()
        
        token_data = response.json()
        access_token = token_data['access_token']
        
        print("✅ Got access token")
        return access_token
        
    except Exception as e:
        print(f"❌ Token request failed: {e}")
        return None

def get_arctic_vessels(access_token, min_latitude=65.0):
    """Get all vessels and filter for Arctic region (above specified latitude)"""
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Accept': 'application/json'
    }
    
    # Get all latest vessel positions
    url = "https://live.ais.barentswatch.no/v1/latest/combined"
    
    print(f"\n🌊 Fetching all vessels from BarentsWatch...")
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        all_vessels = response.json()
        
        # Filter for vessels above the minimum latitude
        arctic_vessels = [
            vessel for vessel in all_vessels 
            if vessel.get('latitude', 0) >= min_latitude
        ]
        
        print(f"✅ Found {len(arctic_vessels)} vessels above {min_latitude}°N")
        print(f"   (Out of {len(all_vessels)} total vessels)")
        
        return arctic_vessels
        
    except Exception as e:
        print(f"❌ Error fetching vessels: {e}")
        return []

def display_arctic_vessels(vessels):
    """Display Arctic vessel information"""
    
    if not vessels:
        print("\n❌ No Arctic vessels found")
        return
    
    print(f"\n🚢 Arctic Vessels (sample of first 10):")
    print("=" * 60)
    
    # Show first 10 vessels as sample
    for vessel in vessels[:10]:
        name = vessel.get('name', 'Unknown')
        mmsi = vessel.get('mmsi', 'N/A')
        lat = vessel.get('latitude', 0)
        lon = vessel.get('longitude', 0)
        ship_type = vessel.get('shipType', 'N/A')
        
        print(f"  • {name:25} MMSI:{mmsi:9} Pos:{lat:.2f}°N, {lon:.2f}°E Type:{ship_type}")
    
    if len(vessels) > 10:
        print(f"\n  ... and {len(vessels) - 10} more vessels")
    
    # Show statistics by ship type
    print(f"\n📊 Statistics by vessel type:")
    ship_types = {}
    for vessel in vessels:
        ship_type = vessel.get('shipType', 'Unknown')
        ship_types[ship_type] = ship_types.get(ship_type, 0) + 1
    
    for ship_type, count in sorted(ship_types.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  Type {ship_type}: {count} vessels")

def main():
    """Main function to get Arctic vessels"""
    print("🗺️  BarentsWatch Arctic Vessel Tracker")
    print("Fetching vessels above 65°N")
    print("=" * 60)
    
    # Step 1: Get access token
    access_token = get_access_token()
    if not access_token:
        print("❌ Failed to get access token. Exiting.")
        return
    
    # Step 2: Get Arctic vessels
    arctic_vessels = get_arctic_vessels(access_token, min_latitude=65.0)
    
    # Step 3: Display results
    display_arctic_vessels(arctic_vessels)
    
    # Optional: Save to JSON file
    if arctic_vessels:
        filename = f"arctic_vessels_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w') as f:
            json.dump(arctic_vessels, f, indent=2)
        print(f"\n💾 Data saved to {filename}")

if __name__ == "__main__":
    main()